In [3]:
import requests
import json

# Replace with your actual Messari API key
MESSARI_API_KEY = ""

# Messari expects the key in the 'x-messari-api-key' header
HEADERS = {
    "x-messari-api-key": MESSARI_API_KEY,
    "Accept": "application/json"
}

def test_messari_key():
    print("🔍 Testing Messari API Key Permissions (Updated Endpoints)...\n")
    
    # Updated 2026 Endpoints
    endpoints = {
        "Basic Market Data (Assets)": "https://api.messari.io/metrics/v2/assets?limit=1",
        "AI Topics & Narratives": "https://api.messari.io/topics/v1/classes",
        "General News Sources": "https://api.messari.io/news/v1/sources",
        "Research Reports (Basic)": "https://api.messari.io/research/v1/reports"
    }

    for name, url in endpoints.items():
        print(f"--- {name} ---")
        try:
            response = requests.get(url, headers=HEADERS, timeout=10)
            
            if response.status_code == 200:
                print("✅ Status: 200 OK (Access Granted)")
                data = response.json()
                preview = json.dumps(data)[:200]
                print(f"📦 Payload Preview: {preview}...\n")
                
            elif response.status_code in [401, 403]:
                print(f"❌ Status: {response.status_code} (Unauthorized/Forbidden)")
                print("Your API key lacks the required subscription tier for this endpoint.\n")
                
            elif response.status_code == 404:
                print(f"❓ Status: 404 (Endpoint Not Found)")
                print("The URL path might still be incorrect or deprecated.\n")
                
            else:
                print(f"❓ Status: {response.status_code}")
                print(f"Response: {response.text[:100]}\n")
                
        except Exception as e:
            print(f"🚨 Request failed: {e}\n")

In [4]:
MESSARI_API_KEY = "8vvfMNaiFAEVv2eN+QKDMyzH12e7se2lpt2pVIAunjXfRhKR"

if MESSARI_API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️ Please insert your API key in the script before running.")
else:
    test_messari_key()

🔍 Testing Messari API Key Permissions (Updated Endpoints)...

--- Basic Market Data (Assets) ---
✅ Status: 200 OK (Access Granted)
📦 Payload Preview: {"error": null, "data": [{"id": "1e31218a-e44e-4285-820c-8282ee222035", "name": "Bitcoin", "slug": "bitcoin", "symbol": "BTC", "category": "Cryptocurrency", "sector": "Cryptocurrency", "sectorV2": ["N...

--- AI Topics & Narratives ---
❌ Status: 403 (Unauthorized/Forbidden)
Your API key lacks the required subscription tier for this endpoint.

--- General News Sources ---
❌ Status: 403 (Unauthorized/Forbidden)
Your API key lacks the required subscription tier for this endpoint.

--- Research Reports (Basic) ---
❌ Status: 403 (Unauthorized/Forbidden)
Your API key lacks the required subscription tier for this endpoint.



In [36]:
from dotenv import load_dotenv
load_dotenv(override=True)

import os
import httpx
from eth_account import Account
from x402 import x402Client
from x402.mechanisms.evm.exact import ExactEvmScheme
from x402.http import x402HTTPClient

PRIVATE_KEY = os.environ.get("WALLET_PRIVATE_KEY")
if not PRIVATE_KEY:
    raise ValueError("Please set the WALLET_PRIVATE_KEY environment variable.")

# Limite augmentée à 1.00 USDC pour autoriser l'accès au News Feed ($0.55)
MAX_SPEND_USDC = 1.00 

async def test_messari_with_budget():
    signer = Account.from_key(PRIVATE_KEY)
    print(f"👛 Loaded Bot Wallet: {signer.address}")

    client = x402Client()
    client.register("eip155:*", ExactEvmScheme(signer=signer))

    endpoints = {
        "Market Data Assets": "https://api.messari.io/metrics/v2/assets",
        "Raw News Feed": "https://api.messari.io/news/v1/news/feed"
    }
    
    # Initialisation du middleware x402
    x402_middleware = x402HTTPClient(client)

    initial_headers = {
        "Accept": "application/json"
    }

    async with httpx.AsyncClient() as http_client:
        for name, url in endpoints.items():
            print(f"\n📡 Probing {name}...")
            try:
                # 1. Premier appel
                response = await http_client.get(url, headers=initial_headers, timeout=15.0)
                
                # Cas nominal : Défi 402
                if response.status_code == 402:
                    payment_data = response.json()
                    
                    # 2. FIX: Extraction correcte du prix (x402 v2)
                    accepts = payment_data.get("accepts", [])
                    if not accepts:
                        print("❌ No payment methods offered by server.")
                        continue

                    first_option = accepts[0]
                    amount_atomic = int(first_option.get("amount", 0))
                    price_usdc = amount_atomic / (10 ** 6)
                    
                    print(f"💰 Quoted Price: ${price_usdc:.4f} USDC")
                    
                    # Vérification du budget
                    if price_usdc > MAX_SPEND_USDC:
                        print(f"🚫 Price exceeds limit (${MAX_SPEND_USDC}). Aborting.")
                        continue
                    
                    print(f"✅ Price accepted. Authorizing Web3 payment...")
                    
                    # 3. Signature du paiement (avec le "await" rajouté)
                    auth_headers, _ = await x402_middleware.handle_402_response(
                        headers=dict(response.headers), 
                        body=response.content
                    )
                    
                    print(f"💳 Payment settled. Fetching data with proof of payment...")
                    
                    # 4. Seconde requête avec preuve de paiement
                    final_response = await http_client.get(url, headers=auth_headers, timeout=15.0)
                    
                    if final_response.status_code == 200:
                        data = final_response.json()
                        print(f"🎉 Success! Retrieved {len(str(data))} bytes of data.")
                        print(f"📦 Preview: {str(data)[:200]}...")
                    else:
                        print(f"❌ Verification failed: HTTP {final_response.status_code}")
                        print(final_response.text[:200])

                elif response.status_code == 200:
                    print("✅ Endpoint is free! Data retrieved.")
                    print(f"📦 Preview: {str(response.json())[:200]}...")
                    
                else:
                    print(f"❌ Server returned HTTP {response.status_code}")
                    print(response.text[:200])

            except Exception as e:
                print(f"🚨 Request Error: {e}")

# Lancement de l'exécution
await test_messari_with_budget()

👛 Loaded Bot Wallet: 0xC78b78eF2547581cC5c685AAF2b394b136B8146a

📡 Probing Market Data Assets...
✅ Endpoint is free! Data retrieved.
📦 Preview: {'error': None, 'data': [{'id': '1e31218a-e44e-4285-820c-8282ee222035', 'name': 'Bitcoin', 'slug': 'bitcoin', 'symbol': 'BTC', 'category': 'Cryptocurrency', 'sector': 'Cryptocurrency', 'sectorV2': ['N...

📡 Probing Raw News Feed...
💰 Quoted Price: $0.5500 USDC
✅ Price accepted. Authorizing Web3 payment...
💳 Payment settled. Fetching data with proof of payment...
❌ Verification failed: HTTP 403
{"error":"x402 verify failed: invalid_payload: contract call failed: unable to call contract: execution reverted","data":null}


In [25]:
await test_messari_with_budget()

👛 Loaded Bot Wallet: 0xC78b78eF2547581cC5c685AAF2b394b136B8146a

📡 Requesting AI Topics & Narratives via x402...
🚨 Request Error: 'x402HTTPClient' object has no attribute 'get'

📡 Requesting Raw News Feed via x402...
🚨 Request Error: 'x402HTTPClient' object has no attribute 'get'


In [27]:
import inspect
import os
from eth_account import Account
from x402 import x402Client
from x402.http import x402HTTPClient

# Initialisation rapide
PRIVATE_KEY = os.environ.get("WALLET_PRIVATE_KEY", "f431e40bb1fe54fc166c92465c3d87c8cc563ad9cfb1f6694f38f9df005e5991")
signer = Account.from_key(PRIVATE_KEY)
client = x402Client()
http_client = x402HTTPClient(client)

print("=== 🔑 SIGNATURES DES MÉTHODES CLES ===")

methods_to_check = [
    'encode_payment_signature_header', 
    'handle_402_response', 
    'create_payment_payload'
]

for method_name in methods_to_check:
    try:
        method = getattr(http_client, method_name)
        print(f"\n👉 {method_name} {inspect.signature(method)}")
    except Exception as e:
        print(f"Impossible de lire {method_name} : {e}")

=== 🔑 SIGNATURES DES MÉTHODES CLES ===

👉 encode_payment_signature_header (payload: 'PaymentPayload | PaymentPayloadV1') -> 'dict[str, str]'

👉 handle_402_response (headers: 'dict[str, str]', body: 'bytes | None') -> 'tuple[dict[str, str], PaymentPayload | PaymentPayloadV1]'

👉 create_payment_payload (payment_required: 'PaymentRequired | PaymentRequiredV1') -> 'PaymentPayload | PaymentPayloadV1'
